# Train NNUE (Kaggle)

In [ ]:
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
batch_size = 4096
num_hidden = 256
learning_rate = 0.001
num_epochs = 40
patience = 4
max_positions = None

csv_file = max(glob.glob("/kaggle/input/**/evals.csv", recursive=True), key=os.path.getsize)
out_dir = "/kaggle/working"
log_file = os.path.join(out_dir, "nnue.log")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print("CSV:", csv_file, "| device:", device)

## Helpers

In [ ]:
def log(text):
    line = f"[{time.strftime('%H:%M:%S')}] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

## Loading the data

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    square = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            square += int(char)
        else:
            row[square] = PIECE_CODES[char]
            square += 1
    return row, color == "w"


def load_compact(csv_file, max_positions):
    if max_positions is None:
        with open(csv_file, encoding="utf-8") as f:
            max_positions = sum(1 for _ in f) - 1

    codes = np.zeros((max_positions, 64), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float32)

    idx = 0
    for chunk in pd.read_csv(csv_file, chunksize=1_000_000, nrows=max_positions):
        start = idx
        for fen in chunk["fen"]:
            codes[idx], side[idx] = parse_fen(fen)
            idx += 1

        cp = chunk["cp"].to_numpy(dtype=np.float64)
        mate = chunk["mate"].to_numpy(dtype=np.float64)
        cp_labels = 2 / (1 + np.exp(-0.00368208 * np.nan_to_num(cp))) - 1
        mate_labels = np.where(mate > 0, 1.0, -1.0)
        labels[start:idx] = np.where(np.isnan(mate), cp_labels, mate_labels)
        log(f"Loaded {idx:,} / {max_positions:,} positions")

    return codes, side, labels

## Features: the board from both perspectives

In [ ]:
NUM_FEATURES = 768

WHITE_FEATURES = torch.full((13, 64), NUM_FEATURES, dtype=torch.long)
BLACK_FEATURES = torch.full((13, 64), NUM_FEATURES, dtype=torch.long)
for code in range(1, 13):
    piece_type = (code - 1) % 6
    is_white = code <= 6
    for index in range(64):
        square = index ^ 56
        WHITE_FEATURES[code, index] = (0 if is_white else 384) + piece_type * 64 + square
        BLACK_FEATURES[code, index] = (384 if is_white else 0) + piece_type * 64 + (square ^ 56)

WHITE_FEATURES = WHITE_FEATURES.to(device)
BLACK_FEATURES = BLACK_FEATURES.to(device)
SQUARES = torch.arange(64, device=device)


def one_hot(table, data_codes):
    indices = table[data_codes.long(), SQUARES]
    x = torch.zeros((data_codes.shape[0], NUM_FEATURES + 1), device=device)
    x.scatter_(1, indices, 1.0)
    return x[:, :NUM_FEATURES]


def features(data_codes, data_side):
    x_white = one_hot(WHITE_FEATURES, data_codes)
    x_black = one_hot(BLACK_FEATURES, data_codes)
    white_to_move = data_side.bool().unsqueeze(1)
    x_us = torch.where(white_to_move, x_white, x_black)
    x_them = torch.where(white_to_move, x_black, x_white)
    return x_us, x_them

## Model

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_features, num_hidden):
        super().__init__()
        self.feature_transformer = nn.Linear(in_features=num_features, out_features=num_hidden)
        self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)
        self.output_act_fn = nn.Tanh()

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us), 0, 1)
        them = torch.clamp(self.feature_transformer(x_them), 0, 1)
        x = torch.cat([us, them], dim=1)
        x = self.output(x)
        x = self.output_act_fn(x)
        return x[:, 0]

## Data

In [ ]:
log(f"Start on {device}, loading data...")
codes, side, labels = load_compact(csv_file, max_positions)

num_positions = len(labels)
perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).numpy()
num_train = int(0.8 * num_positions)
num_val = int(0.1 * num_positions)
train_idx = perm[:num_train]
val_idx = perm[num_train:num_train + num_val]
test_idx = perm[num_train + num_val:]
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")


def batches(indices, shuffle):
    if shuffle:
        indices = indices[torch.randperm(len(indices)).numpy()]
    for start in range(0, len(indices), batch_size):
        batch = np.sort(indices[start:start + batch_size])
        data_codes = torch.from_numpy(codes[batch]).to(device)
        data_side = torch.from_numpy(side[batch]).to(device)
        data_labels = torch.from_numpy(labels[batch]).to(device)
        x_us, x_them = features(data_codes, data_side)
        data_labels = torch.where(data_side.bool(), data_labels, -data_labels)
        yield x_us, x_them, data_labels

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def train_one_epoch(model, optimizer, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, data_labels in batches(indices, shuffle=True):
        preds = model(x_us, x_them)
        loss = loss_fn(preds, data_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def eval_model(model, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, data_labels in batches(indices, shuffle=False):
            preds = model(x_us, x_them)

            total_loss += loss_fn(preds, data_labels).item()
            num_batches += 1

            total_abs_error += (preds - data_labels).abs().sum().item()
            num_preds += data_labels.shape[0]

    return total_loss / num_batches, total_abs_error / num_preds

## Training

In [ ]:
model = NNUE(num_features=NUM_FEATURES, num_hidden=num_hidden).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.3, patience=1)
model_path = os.path.join(out_dir, "nnue.tar")
print(model)

history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(num_epochs):
    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, train_idx)
    val_loss, val_mae = eval_model(model, val_idx)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), model_path)
        log("Checkpoint saved")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            log(f"No improvement for {patience} epochs, stopping")
            break

## Test

In [ ]:
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))
test_loss, test_mae = eval_model(model, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f}")

params = {"num_features": NUM_FEATURES, "num_hidden": num_hidden, "batch_size": batch_size, "learning_rate": learning_rate}
with open(os.path.join(out_dir, "nnue.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae}, f, indent=2)
log("Done")

## Sanity check

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white is a queen up, white to move": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white is a queen up, black to move": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR b KQkq - 0 1",
}

model.eval()
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    data_codes = torch.from_numpy(row).unsqueeze(0).to(device)
    data_side = torch.tensor([white], dtype=torch.uint8, device=device)
    with torch.no_grad():
        value = model(*features(data_codes, data_side)).item()
    log(f"{name}: {value:+.3f} for the side to move")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_curves.png"), bbox_inches="tight")
plt.show()